# CosyVoice3 Fine-Tuning (Debi & Marlene)

Base Model: `Fun-CosyVoice3-0.5B` (Qwen2.5-0.5B LLM backbone)

- LLM SFT only (Flow/HiFiGAN pretrained 유지)
- Multi-speaker: debi + marlene (CAMPlus speaker embedding)
- 4 style tags: `[excited]`, `[sad]`, `[happy]`, `[calm]`

## Pipeline
1. Drive 마운트 + 데이터 복사
2. CosyVoice repo clone + 의존성
3. Pretrained 모델 다운로드
4. wav.scp 경로 변환 + 데이터 링크
5. Speaker Embedding 추출 (campplus)
6. Speech Token 추출 (speech_tokenizer_v3)
7. Parquet 변환
8. Config 수정 (fine-tune용 LR/epoch)
9. Dry Run (1 epoch)
10. 본 학습 (LLM SFT)
11. Checkpoint 평균화 + 음성 평가
12. Drive/HuggingFace 저장

---
## 1. Google Drive Mount + GPU Check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

import os
WORK_DIR = '/content/CosyVoice'
DATA_DIR = '/content/cosyvoice3_data'
BACKUP_DIR = '/content/drive/MyDrive/cosyvoice3_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f'Work: {WORK_DIR}')
print(f'Data: {DATA_DIR}')
print(f'Backup: {BACKUP_DIR}')

---
## 2. 데이터 업로드

`cosyvoice3_data/`를 **zip으로 압축**해서 Drive에 올리세요.
Drive FUSE 마운트는 수백 개 작은 파일 복사 시 불안정합니다.

```
# 로컬에서 실행:
cd tts_training
zip -r cosyvoice3_data.zip cosyvoice3_data/
# -> cosyvoice3_data.zip을 Google Drive에 업로드
```

In [ ]:
import os
import shutil

DATA_DIR = '/content/cosyvoice3_data'

# === Method A: Drive에 zip 파일이 있으면 unzip (권장) ===
DRIVE_ZIP = '/content/drive/MyDrive/cosyvoice3_data.zip'

if os.path.exists(DRIVE_ZIP):
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    !unzip -q "{DRIVE_ZIP}" -d /content/
    print(f'Unzipped: {DRIVE_ZIP} -> {DATA_DIR}')

# === Method B: Drive에 폴더가 있으면 파일별 복사 (retry 포함) ===
elif os.path.exists('/content/drive/MyDrive/cosyvoice3_data'):
    import time
    DRIVE_DATA = '/content/drive/MyDrive/cosyvoice3_data'
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)

    # 디렉토리 구조 먼저 생성
    for root, dirs, files in os.walk(DRIVE_DATA):
        rel = os.path.relpath(root, DRIVE_DATA)
        os.makedirs(os.path.join(DATA_DIR, rel), exist_ok=True)

    # 파일별 복사 (실패 시 3회 retry)
    failed = []
    for root, dirs, files in os.walk(DRIVE_DATA):
        rel = os.path.relpath(root, DRIVE_DATA)
        for fname in files:
            src = os.path.join(root, fname)
            dst = os.path.join(DATA_DIR, rel, fname)
            for attempt in range(3):
                try:
                    shutil.copy2(src, dst)
                    break
                except Exception as e:
                    if attempt < 2:
                        time.sleep(1)
                    else:
                        failed.append(f'{fname}: {e}')

    if failed:
        print(f'WARNING: {len(failed)} files failed:')
        for f in failed[:10]:
            print(f'  {f}')
    else:
        print(f'Drive folder -> {DATA_DIR} 복사 완료')

# === Method C: 직접 업로드 ===
else:
    print('Drive에 데이터 없음 - zip 직접 업로드')
    from google.colab import files
    uploaded = files.upload()  # cosyvoice3_data.zip 선택
    !unzip -q cosyvoice3_data.zip -d /content/

# 데이터 확인
print('')
for split in ['train', 'dev']:
    split_dir = os.path.join(DATA_DIR, split)
    if os.path.exists(split_dir):
        wavs_dir = os.path.join(split_dir, 'wavs')
        wav_count = len([f for f in os.listdir(wavs_dir) if f.endswith('.wav')])
        print(f'{split}: {wav_count} wavs')
        with open(os.path.join(split_dir, 'text'), 'r') as f:
            lines = f.readlines()
            print(f'  text: {len(lines)} lines')
            print(f'  sample: {lines[0].strip()[:120]}')

---
## 3. CosyVoice Repo Clone + 의존성 설치

In [ ]:
%%bash
cd /content

if [ ! -d "CosyVoice" ]; then
    echo "Cloning CosyVoice..."
    git clone https://github.com/FunAudioLLM/CosyVoice.git
    cd CosyVoice
    git submodule update --init --recursive
else
    echo "CosyVoice already exists"
fi

# 항상 /content/CosyVoice로 이동 (if/else 어느 분기든 동일)
cd /content/CosyVoice

# path.sh 생성 (run.sh 첫 줄에서 `. ./path.sh || exit 1` 로 필수 로드)
mkdir -p examples/libritts/cosyvoice3
cat > examples/libritts/cosyvoice3/path.sh << 'PATHEOF'
export PYTHONPATH=/content/CosyVoice/third_party/Matcha-TTS:/content/CosyVoice:${PYTHONPATH:-}
PATHEOF

echo ""
echo "=== path.sh ==="
cat examples/libritts/cosyvoice3/path.sh
echo ""
echo "=== Submodules ==="
git submodule status 2>&1 | head -5
echo ""
echo "=== Key directories ==="
ls -d tools/ cosyvoice/ third_party/Matcha-TTS/ examples/libritts/cosyvoice3/ 2>&1

In [ ]:
%%bash
cd /content/CosyVoice

# Colab에 이미 PyTorch + CUDA가 있으므로 나머지만 설치
pip install -q conformer==0.3.2 \
    deepspeed==0.14.5 \
    diffusers==0.32.2 \
    gdown \
    grpcio==1.63.0 grpcio-tools==1.63.0 \
    hydra-core==1.3.2 \
    HyperPyYAML==1.2.2 \
    inflect==7.3.1 \
    librosa==0.10.2.post1 \
    lightning==2.4.0 \
    matplotlib==3.9.2 \
    modelscope==1.22.3 \
    networkx==3.4.2 \
    omegaconf==2.3.0 \
    onnxruntime-gpu==1.19.0 \
    openai-whisper==20240930 \
    protobuf==5.28.3 \
    pydantic==2.9.2 \
    pyyaml \
    rich==13.9.4 \
    soundfile==0.12.1 \
    tensorboard==2.18.0 \
    transformers==4.48.2 \
    wget==3.2

pip install -q WeTextProcessing==1.0.4.1 || echo 'WeTextProcessing failed - not critical'

cd /content/CosyVoice/third_party/Matcha-TTS
pip install -q -e . 2>/dev/null || echo 'Matcha-TTS skipped'

echo '=== Done ==='

---
## 4. Pretrained 모델 다운로드

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_DIR = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'

if not os.path.exists(MODEL_DIR) or not os.listdir(MODEL_DIR):
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    snapshot_download(
        repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.md', '*.txt'],
    )
    print('Model downloaded')
else:
    print(f'Model exists: {MODEL_DIR}')

# 필수 파일 확인
required = ['llm.pt', 'flow.pt', 'hift.pt', 'campplus.onnx',
            'speech_tokenizer_v3.onnx', 'cosyvoice3.yaml']
for f in required:
    path = os.path.join(MODEL_DIR, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    status = f'{size:.1f}MB' if exists else 'MISSING'
    print(f'  {f}: {status}')

# CosyVoice-BlankEN 디렉토리 확인 (qwen_pretrain_path에 필요)
blanken = os.path.join(MODEL_DIR, 'CosyVoice-BlankEN')
print(f'  CosyVoice-BlankEN/: {"OK" if os.path.isdir(blanken) else "MISSING"}')

---
## 5. wav.scp 경로 변환 + 데이터 링크

로컬에서 생성한 `wav.scp`의 상대경로를 Colab 절대경로로 변환합니다.

In [ ]:
import os

DATA_DIR = '/content/cosyvoice3_data'

for split in ['train', 'dev']:
    split_dir = os.path.join(DATA_DIR, split)
    scp_path = os.path.join(split_dir, 'wav.scp')

    with open(scp_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split(' ', 1)
        if len(parts) != 2:
            continue
        utt_id, wav_path = parts
        # 상대경로 'wavs/xxx.wav' -> 절대경로
        if not wav_path.startswith('/'):
            wav_path = os.path.join(split_dir, wav_path)
        # Windows 경로가 들어왔을 경우에도 처리
        if '\\' in wav_path or ':' in wav_path:
            wav_path = os.path.join(split_dir, 'wavs', os.path.basename(wav_path))
        new_lines.append(f'{utt_id} {wav_path}\n')

    with open(scp_path, 'w') as f:
        f.writelines(new_lines)

    print(f'{split}/wav.scp: {len(new_lines)} entries')
    # 첫 줄 확인
    print(f'  sample: {new_lines[0].strip()}')
    # 파일 존재 확인
    sample_wav = new_lines[0].strip().split(' ', 1)[1]
    print(f'  exists: {os.path.exists(sample_wav)}')

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

# 학습 디렉토리에 심볼릭 링크
mkdir -p data

ln -sfn /content/cosyvoice3_data/train data/train
ln -sfn /content/cosyvoice3_data/dev data/dev

# 확인
echo '--- train ---'
wc -l data/train/wav.scp data/train/text data/train/utt2spk
echo '--- dev ---'
wc -l data/dev/wav.scp data/dev/text data/dev/utt2spk

echo ''
echo '--- sample wav.scp (첫 2줄) ---'
head -2 data/train/wav.scp
echo ''
echo '--- sample text (첫 2줄) ---'
head -2 data/train/text

# wav 파일 존재 확인
FIRST_WAV=$(head -1 data/train/wav.scp | awk '{print $2}')
if [ -f "$FIRST_WAV" ]; then
    echo ""
    echo "wav file check: OK ($FIRST_WAV)"
else
    echo ""
    echo "WARNING: wav not found: $FIRST_WAV"
    echo "wav.scp 경로가 절대경로인지 확인하세요 (Cell 5 실행 필요)"
fi

---
## 6. Speaker Embedding 추출 (CAMPlus)

CPU에서 실행됩니다. `campplus.onnx`로 192-dim speaker embedding을 추출합니다.

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Embedding: $split ==="
    python ../../../tools/extract_embedding.py \
        --dir data/$split \
        --onnx_path $PRETRAINED/campplus.onnx \
        --num_thread 4
done

echo ''
for split in train dev; do
    echo "$split:"
    ls -lh data/$split/utt2embedding.pt data/$split/spk2embedding.pt 2>/dev/null || echo "  MISSING!"
done

---
## 7. Speech Token 추출 (speech_tokenizer_v3)

GPU(CUDA) 필요. 6561-vocab speech token을 추출합니다.

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Speech Token: $split ==="
    python ../../../tools/extract_speech_token.py \
        --dir data/$split \
        --onnx_path $PRETRAINED/speech_tokenizer_v3.onnx
done

echo ''
for split in train dev; do
    echo "$split:"
    ls -lh data/$split/utt2speech_token.pt 2>/dev/null || echo "  MISSING!"
done

---
## 8. Parquet 변환

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

for split in train dev; do
    echo "=== Parquet: $split ==="
    mkdir -p data/$split/parquet
    python ../../../tools/make_parquet_list.py \
        --num_utts_per_parquet 1000 \
        --num_processes 4 \
        --src_dir data/$split \
        --des_dir data/$split/parquet
    ls -lh data/$split/parquet/
done

# data.list 생성 (상대경로 - train.py도 같은 디렉토리에서 실행)
cat data/train/parquet/data.list > data/train.data.list
cat data/dev/parquet/data.list > data/dev.data.list

echo ''
echo 'train.data.list:'
cat data/train.data.list
echo ''
echo 'dev.data.list:'
cat data/dev.data.list

---
## 9. Config 준비

기존 `cosyvoice3.yaml`의 train_conf를 fine-tuning용으로 수정합니다.
- `max_epoch: 50` (30-40에서 best 예상)
- `log_interval: 10`

DeepSpeed config도 생성합니다 (repo에 있으면 그걸 사용).

In [ ]:
import os
import yaml

TRAIN_BASE = '/content/CosyVoice/examples/libritts/cosyvoice3'
CONF_DIR = os.path.join(TRAIN_BASE, 'conf')
os.makedirs(CONF_DIR, exist_ok=True)

# cosyvoice3.yaml 가져오기 (examples/conf 또는 pretrained에서)
config_path = os.path.join(CONF_DIR, 'cosyvoice3.yaml')
pretrained_config = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B/cosyvoice3.yaml'

if not os.path.exists(config_path):
    if os.path.exists(pretrained_config):
        import shutil
        shutil.copy2(pretrained_config, config_path)
        print(f'Config copied from pretrained: {config_path}')
    else:
        print('ERROR: cosyvoice3.yaml not found anywhere!')
        print(f'Checked: {config_path}')
        print(f'Checked: {pretrained_config}')
else:
    print(f'Config exists: {config_path}')

# config 읽어서 train_conf 수정
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()

    # HyperPyYAML 형식이라 일반 yaml.load로 안 될 수 있음
    # 그냥 텍스트 치환으로 안전하게 수정
    print('\n--- train_conf 부분 ---')
    in_train_conf = False
    for line in content.split('\n'):
        if 'train_conf:' in line:
            in_train_conf = True
        if in_train_conf:
            print(line)
            if line.strip() and not line.startswith(' ') and 'train_conf' not in line:
                break

    print('\n(Config는 HyperPyYAML 형식이라 수동 수정이 안전합니다)')
    print('train.py의 YAML이 max_epoch, log_interval을 내부에서 읽습니다.')
    print('필요시 위 값들을 직접 수정하세요.')

In [ ]:
import json
import os

TRAIN_BASE = '/content/CosyVoice/examples/libritts/cosyvoice3'
CONF_DIR = os.path.join(TRAIN_BASE, 'conf')

# repo에 이미 ds_stage2.json이 있으면 그걸 사용
existing_ds = os.path.join(CONF_DIR, 'ds_stage2.json')
if os.path.exists(existing_ds):
    with open(existing_ds, 'r') as f:
        ds_config = json.load(f)
    print(f'Using existing DeepSpeed config: {existing_ds}')
    print(json.dumps(ds_config, indent=2))
else:
    # CosyVoice의 train.py가 optimizer/scheduler를 내부에서 관리하므로
    # DeepSpeed config에는 ZeRO 최적화만 넣고 optimizer는 빼야 함
    ds_config = {
        "train_micro_batch_size_per_gpu": "auto",
        "gradient_accumulation_steps": "auto",
        "fp16": {
            "enabled": "auto"
        },
        "zero_optimization": {
            "stage": 2,
            "allgather_partitions": True,
            "allgather_bucket_size": 5e8,
            "overlap_comm": True,
            "reduce_scatter": True,
            "reduce_bucket_size": 5e8,
            "contiguous_gradients": True
        },
        "gradient_clipping": 5.0,
        "steps_per_print": 10,
        "wall_clock_breakdown": False
    }
    with open(existing_ds, 'w') as f:
        json.dump(ds_config, f, indent=2)
    print(f'DeepSpeed config created: {existing_ds}')
    print(json.dumps(ds_config, indent=2))

---
## 10. Dry Run (1 Epoch 검증)

본 학습 전에 1 epoch만 돌려서 확인:
- loss가 정상 출력되는지
- OOM 없이 GPU 메모리가 안정적인지
- `<|endofprompt|>` / 스타일 태그가 정상 처리되는지

**참고**: `--qwen_pretrain_path`는 pretrained 모델 내 `CosyVoice-BlankEN` 디렉토리를 가리킵니다.

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"
CONFIG="conf/cosyvoice3.yaml"

echo "=== DRY RUN: 1 epoch ==="
echo "Config: $CONFIG"
echo "Pretrained: $PRETRAINED"
echo ""

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config $CONFIG \
  --train_data data/train.data.list \
  --cv_data data/dev.data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/dryrun/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/dryrun \
  --ddp.dist_backend nccl \
  --num_workers 2 \
  --prefetch 100 \
  --pin_memory \
  --use_amp \
  --deepspeed_config ./conf/ds_stage2.json \
  --deepspeed.save_states model+optimizer \
  2>&1 | tail -80

echo ""
echo "=== GPU Memory ==="
nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

---
## 11. 본 학습 (LLM SFT)

Dry run이 성공했으면 본 학습을 실행합니다.

- LLM 파트만 학습 (Flow/HiFiGAN freeze)
- 5분 주기 자동 백업
- 학습 로그 -> 시각화

**중요**: `max_epoch`은 config YAML에서 결정됩니다. Dry run에서 config 내용을 확인하세요.

In [ ]:
import threading
import time
import shutil
import os

stop_backup = threading.Event()

def auto_backup():
    EXP_DIR = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3/llm/torch_ddp'
    BACKUP_EXP = '/content/drive/MyDrive/cosyvoice3_backup/exp'
    while not stop_backup.is_set():
        time.sleep(300)
        if os.path.exists(EXP_DIR):
            try:
                pts = sorted(
                    [f for f in os.listdir(EXP_DIR) if f.endswith('.pt')],
                    key=lambda x: os.path.getmtime(os.path.join(EXP_DIR, x)),
                    reverse=True
                )
                if pts:
                    os.makedirs(BACKUP_EXP, exist_ok=True)
                    src = os.path.join(EXP_DIR, pts[0])
                    dst = os.path.join(BACKUP_EXP, pts[0])
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
                        print(f'[Backup] {pts[0]} -> Drive')
            except Exception as e:
                print(f'[Backup Error] {e}')

backup_thread = threading.Thread(target=auto_backup, daemon=True)
backup_thread.start()
print('Auto backup started (5min interval)')

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"
CONFIG="conf/cosyvoice3.yaml"

echo "=== LLM SFT Training ==="
echo "Config: $CONFIG"
echo "Pretrained: $PRETRAINED"
echo ""

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config $CONFIG \
  --train_data data/train.data.list \
  --cv_data data/dev.data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/cosyvoice3/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/cosyvoice3/llm/torch_ddp \
  --ddp.dist_backend nccl \
  --num_workers 2 \
  --prefetch 100 \
  --pin_memory \
  --use_amp \
  --deepspeed_config ./conf/ds_stage2.json \
  --deepspeed.save_states model+optimizer \
  2>&1 | tee /content/train_log.txt

In [ ]:
stop_backup.set()

import re
import matplotlib.pyplot as plt
import os

train_losses = []
cv_losses = []

log_path = '/content/train_log.txt'
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        for line in f:
            m = re.search(r'train_loss[=:\s]+(\d+\.\d+)', line)
            if m:
                train_losses.append(float(m.group(1)))
            m = re.search(r'cv_loss[=:\s]+(\d+\.\d+)', line)
            if m:
                cv_losses.append(float(m.group(1)))

if train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(train_losses)
    axes[0].set_title('Train Loss')
    axes[0].set_xlabel('Log Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)

    if cv_losses:
        axes[1].plot(cv_losses, color='orange')
        axes[1].set_title('CV Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].grid(True)
        if len(cv_losses) >= 3 and cv_losses[-1] > cv_losses[-2] > cv_losses[-3]:
            print('*** WARNING: CV loss increasing 3 epochs in a row - overfitting! ***')

    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=100)
    plt.show()
    print(f'Train: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}')
    if cv_losses:
        print(f'CV: {cv_losses[0]:.4f} -> {cv_losses[-1]:.4f} (best: {min(cv_losses):.4f})')
else:
    print('No loss found in log - check train_log.txt')

---
## 12. Checkpoint 평균화 + 음성 평가

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

EXP_DIR="exp/cosyvoice3/llm/torch_ddp"

echo "=== Available checkpoints ==="
ls -lh $EXP_DIR/*.pt 2>/dev/null | tail -10
echo ""

echo "=== Checkpoint Averaging (Top 5, val_best) ==="
python ../../../cosyvoice/bin/average_model.py \
    --dst_model $EXP_DIR/llm.pt \
    --src_path $EXP_DIR \
    --num 5 \
    --val_best

echo ""
ls -lh $EXP_DIR/llm.pt

In [ ]:
import sys
import os
import shutil
sys.path.insert(0, '/content/CosyVoice')

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
EXP_DIR = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3/llm/torch_ddp'

# 원본 백업 + 파인튜닝된 LLM으로 교체
llm_orig = os.path.join(PRETRAINED, 'llm.pt')
llm_bak = os.path.join(PRETRAINED, 'llm.pt.bak')
llm_ft = os.path.join(EXP_DIR, 'llm.pt')

if not os.path.exists(llm_bak):
    shutil.copy2(llm_orig, llm_bak)
    print(f'Original backed up: {llm_bak}')
shutil.copy2(llm_ft, llm_orig)
print(f'Fine-tuned LLM applied: {llm_ft} -> {llm_orig}')

# CosyVoice3 모델 로드 (load_jit 없음 - v3에서 제거됨)
from cosyvoice.cli.cosyvoice import CosyVoice3
cosyvoice = CosyVoice3(PRETRAINED, load_trt=False)
print('Model loaded')

In [ ]:
import os
import soundfile as sf
from IPython.display import Audio, display

test_sentences = [
    ("나한텐 일상이었어", "[excited]"),
    ("미안해, 다음엔 더 잘할게", "[sad]"),
    ("오~ 꽤하잖아!", "[happy]"),
    ("한숨 돌렸다 가자고. 시간은 충분하니까", "[calm]"),
    ("안녕하세요, 데비입니다. 오늘도 좋은 하루 되세요.", "[calm]"),
]

output_dir = '/content/test_outputs'
os.makedirs(output_dir, exist_ok=True)

for i, (text, style) in enumerate(test_sentences):
    print(f'\n--- Test {i+1}: {style} "{text}" ---')

    # instruct_text: 학습 때 text 앞에 붙었던 instruction
    # tts_text: [style]태그 + 실제 텍스트 (학습 데이터의 <|endofprompt|> 뒤 부분)
    instruct_text = f'You are a helpful assistant.<|endofprompt|>'
    tts_text = f'{style}{text}'

    for speaker in ['debi', 'marlene']:
        try:
            # 레퍼런스 오디오 (화자 목소리 제공)
            wavs_dir = '/content/cosyvoice3_data/train/wavs/'
            spk_wavs = sorted([f for f in os.listdir(wavs_dir) if f.startswith(speaker)])
            if not spk_wavs:
                print(f'  {speaker}: no reference wav found')
                continue
            ref_wav = os.path.join(wavs_dir, spk_wavs[0])

            output_wav = None
            for result in cosyvoice.inference_instruct2(
                tts_text,
                instruct_text,
                ref_wav,
                stream=False,
            ):
                output_wav = result['tts_speech'].numpy()

            if output_wav is not None:
                out_path = f'{output_dir}/test_{i+1}_{speaker}_{style.strip("[]")}.wav'
                sf.write(out_path, output_wav.flatten(), 24000)
                print(f'  {speaker}: {out_path}')
                display(Audio(output_wav.flatten(), rate=24000))
        except Exception as e:
            print(f'  {speaker}: ERROR - {e}')

---
## 13. 체크포인트 저장 (Drive + HuggingFace Hub)

In [ ]:
import shutil
import os

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
SAVE_DIR = '/content/drive/MyDrive/cosyvoice3_backup/final_model'
os.makedirs(SAVE_DIR, exist_ok=True)

# 추론에 필요한 파일만 복사
files_to_save = [
    'llm.pt',
    'flow.pt',
    'hift.pt',
    'campplus.onnx',
    'speech_tokenizer_v3.onnx',
    'cosyvoice3.yaml',
]

for fname in files_to_save:
    src = os.path.join(PRETRAINED, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SAVE_DIR, fname))
        size_mb = os.path.getsize(src) / (1024*1024)
        print(f'  {fname}: {size_mb:.1f} MB')
    else:
        print(f'  {fname}: not found')

# CosyVoice-BlankEN (Qwen tokenizer) 복사
for d in os.listdir(PRETRAINED):
    src = os.path.join(PRETRAINED, d)
    if os.path.isdir(src):
        dst = os.path.join(SAVE_DIR, d)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  {d}/ (dir)')

# 레퍼런스 오디오도 저장 (Modal 배포 시 필요)
ref_dir = os.path.join(SAVE_DIR, 'references')
os.makedirs(ref_dir, exist_ok=True)
wavs_dir = '/content/cosyvoice3_data/train/wavs/'
for speaker in ['debi', 'marlene']:
    spk_wavs = sorted([f for f in os.listdir(wavs_dir) if f.startswith(speaker)])
    if spk_wavs:
        src = os.path.join(wavs_dir, spk_wavs[0])
        shutil.copy2(src, os.path.join(ref_dir, f'{speaker}.wav'))
        print(f'  references/{speaker}.wav')

print(f'\nSaved to: {SAVE_DIR}')

In [ ]:
from huggingface_hub import HfApi, login

# HF 로그인 (토큰 입력 필요)
login()

HF_REPO = '2R4mi/cosyvoice3-debi-marlene'
SAVE_DIR = '/content/drive/MyDrive/cosyvoice3_backup/final_model'

api = HfApi()
api.create_repo(repo_id=HF_REPO, exist_ok=True, private=True)

api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    commit_message='CosyVoice3 fine-tuned: debi + marlene (LLM SFT)',
)
print(f'Uploaded: https://huggingface.co/{HF_REPO}')